# Notebook 4: Deep Learning Benchmarks (PyTorch)

**Course:** DS 285 — Numerical Methods for Data Science  
**Author:** Rajneesh Babu · M.Tech CDS · IISc Bengaluru

---

Benchmark all optimizers on training a small MLP on synthetic data using PyTorch.
This validates the NumPy theory on a real non-convex neural network loss surface.

**Experiment:**
- Task: Binary classification on sklearn's `make_moons`
- Model: 3-layer MLP (32→64→32→1)
- Fixed 200 epochs per optimizer
- Compare: final loss, convergence speed, stability

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

plt.style.use('dark_background')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.manual_seed(42)
np.random.seed(42)

## 1. Dataset

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1).to(device)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 2. Model Definition

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model parameters: {count_params(MLP()):,}')

## 3. Training Loop

In [ ]:
def train_model(optimizer_fn, n_epochs=200, batch_size=64):
    """
    Train MLP and return train/val loss history.

    Parameters
    ----------
    optimizer_fn : callable(params) -> torch.optim.Optimizer
    """
    torch.manual_seed(42)
    model = MLP().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optimizer_fn(model.parameters())

    n_train = len(X_train_t)
    train_losses, val_losses = [], []

    for epoch in range(n_epochs):
        model.train()
        perm = torch.randperm(n_train)
        epoch_loss = 0.0
        n_batches = 0

        for i in range(0, n_train, batch_size):
            idx = perm[i:i+batch_size]
            xb, yb = X_train_t[idx], y_train_t[idx]
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        train_losses.append(epoch_loss / n_batches)

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_test_t), y_test_t).item()
        val_losses.append(val_loss)

    # Final accuracy
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X_test_t)) > 0.5).float()
        acc = (preds == y_test_t).float().mean().item()

    return train_losses, val_losses, acc

In [ ]:
optimizers_torch = {
    'SGD':            lambda p: optim.SGD(p, lr=0.05),
    'SGD+Momentum':   lambda p: optim.SGD(p, lr=0.05, momentum=0.9),
    'SGD+Nesterov':   lambda p: optim.SGD(p, lr=0.05, momentum=0.9, nesterov=True),
    'Adagrad':        lambda p: optim.Adagrad(p, lr=0.1),
    'RMSprop':        lambda p: optim.RMSprop(p, lr=0.01),
    'Adam':           lambda p: optim.Adam(p, lr=0.001),
}

COLORS_TORCH = ['#60a5fa', '#34d399', '#fbbf24', '#f87171', '#a78bfa', '#fb923c']

all_results = {}
for (name, opt_fn), color in zip(optimizers_torch.items(), COLORS_TORCH):
    train_l, val_l, acc = train_model(opt_fn, n_epochs=200)
    all_results[name] = {'train': train_l, 'val': val_l, 'acc': acc, 'color': color}
    print(f'{name:<16}: final train loss = {train_l[-1]:.4f}, test acc = {acc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0e1a')

for ax, loss_key, title in zip(
    axes,
    ['train', 'val'],
    ['Training Loss', 'Validation Loss']
):
    ax.set_facecolor('#0f172a')
    for name, data in all_results.items():
        ax.plot(data[loss_key], label=name, color=data['color'], linewidth=2)
    ax.set_title(f'MLP — {title}', color='white', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch', color='#94a3b8')
    ax.set_ylabel('BCE Loss', color='#94a3b8')
    ax.tick_params(colors='#64748b')
    for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
    ax.grid(True, alpha=0.15, color='#334155')

axes[0].legend(facecolor='#1e293b', edgecolor='#334155', labelcolor='white', fontsize=9)
plt.suptitle('Deep Learning Optimizer Benchmark — make_moons',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/dl_benchmark.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 4. Test Accuracy Bar Chart

In [ ]:
names = list(all_results.keys())
accs  = [all_results[n]['acc'] for n in names]
colors = [all_results[n]['color'] for n in names]

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_facecolor('#0f172a')
fig.patch.set_facecolor('#0a0e1a')

bars = ax.bar(names, accs, color=colors, edgecolor='none', width=0.6)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.3f}', ha='center', va='bottom', color='white', fontsize=9)

ax.set_ylim(0.8, 1.01)
ax.set_title('Test Accuracy per Optimizer', color='white', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', color='#94a3b8')
ax.tick_params(colors='#64748b')
ax.set_xticklabels(names, rotation=20, ha='right', color='white')
for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
ax.grid(True, axis='y', alpha=0.15, color='#334155')
plt.tight_layout()
plt.savefig('../results/figures/dl_accuracy_bar.png',
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## Key Findings

| Optimizer | Convergence Speed | Final Loss | Generalisation |
|-----------|------------------|-----------|----------------|
| SGD | Slow | High | Good |
| SGD+Momentum | Medium | Medium | Good |
| SGD+Nesterov | Fast | Low | Best |
| Adagrad | Medium | Medium | Moderate |
| RMSprop | Fast | Low | Good |
| Adam | Fastest | Lowest | Good |

**Conclusion:** Adam converges fastest in deep learning; SGD+Nesterov generalises best and matches Adam in final accuracy, consistent with the well-known observation that adaptive methods converge faster but may not generalise better on small datasets.